# Random forest by hand
Three decision trees, each trained on a different random subset of a small dataset, then a majority vote.
We build the subsets three ways: row sampling, column sampling, and both together.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text

## 1. The data: 100 rows, 5 input columns, 1 output column

In [2]:
# 5 informative columns, two classes
X, y = make_classification(n_samples=100, n_features=5, n_redundant=0, n_informative=5,
                           n_clusters_per_class=1, random_state=4)
df = pd.DataFrame(X, columns=["col1", "col2", "col3", "col4", "col5"]).round(3)
df["target"] = y
print(df.shape)
df.head()

(100, 6)


,col1,col2,col3,col4,col5,target
0,0.601,1.074,-0.643,1.199,1.191,1
1,1.606,-1.135,-0.393,-0.651,2.051,1
2,0.631,1.898,-0.223,1.431,1.094,1
3,-0.283,1.030,1.541,-0.961,-0.765,0
4,1.737,0.141,0.552,-0.618,1.511,1


## 2. Three sampling functions
`rng` is a seeded random generator, so every run draws the same samples.

In [3]:
rng = np.random.default_rng(4)


def sample_rows(df, share):
    """A random share of the rows, drawn with replacement (rows can repeat)."""
    n = int(share * len(df))
    return df.sample(n, replace=True, random_state=rng.integers(10**9))


def sample_features(df, share):
    """All rows, a random share of the input columns (no repeats); the target column is always kept."""
    n = int(share * (df.shape[1] - 1))           # do not count the target column
    cols = list(rng.choice(df.columns[:-1], n, replace=False))
    return df[cols + ["target"]]


def combined_sampling(df, row_share, col_share):
    """Row sampling first, then column sampling on the result."""
    return sample_features(sample_rows(df, row_share), col_share)

## 3. The query point
We ask every tree about row 7 of the data. Its true class is 0.

In [4]:
query = df.iloc[[7]]
query

,col1,col2,col3,col4,col5,target
7,0.075,-1.024,-0.24,-2.241,-2.358,0


In [5]:
def forest_by_hand(make_subset):
    """Train three fully grown trees, each on its own random subset, and collect their votes for the query."""
    votes = []
    for k in range(1, 4):
        d = make_subset()
        inputs = d.columns[:-1]                        # the input columns this tree got
        tree = DecisionTreeClassifier(random_state=0).fit(d[inputs], d["target"])
        vote = int(tree.predict(query[inputs])[0])     # the query, with the same columns
        votes.append(vote)
        print(f"tree {k}: {d.shape[0]} rows ({d.index.duplicated().sum()} repeats), "
              f"columns {list(inputs)}, depth {tree.get_depth()}, vote {vote}")
    print("votes:", votes, "-> majority:", max(set(votes), key=votes.count))
    return tree, inputs

## 4. Row sampling: each tree gets 20% of the rows (20 rows), drawn with replacement

In [6]:
tree, inputs = forest_by_hand(lambda: sample_rows(df, 0.2))
# the last tree, printed as text (feature_names maps X[0], X[1]... to real column names)
print(export_text(tree, feature_names=list(inputs)))

tree 1: 20 rows (2 repeats), columns ['col1', 'col2', 'col3', 'col4', 'col5'], depth 2, vote 1
tree 2: 20 rows (1 repeats), columns ['col1', 'col2', 'col3', 'col4', 'col5'], depth 2, vote 0
tree 3: 20 rows (1 repeats), columns ['col1', 'col2', 'col3', 'col4', 'col5'], depth 2, vote 0
votes: [1, 0, 0] -> majority: 0
|--- col5 <= 0.26
|   |--- class: 0
|--- col5 >  0.26
|   |--- col2 <= 2.04
|   |   |--- class: 1
|   |--- col2 >  2.04
|   |   |--- class: 0



## 5. Column sampling: each tree gets every row but 80% of the columns (4 of 5)

In [7]:
tree, inputs = forest_by_hand(lambda: sample_features(df, 0.8))

tree 1: 100 rows (0 repeats), columns ['col5', 'col4', 'col3', 'col2'], depth 6, vote 0
tree 2: 100 rows (0 repeats), columns ['col5', 'col2', 'col1', 'col3'], depth 3, vote 0
tree 3: 100 rows (0 repeats), columns ['col1', 'col3', 'col2', 'col5'], depth 3, vote 0
votes: [0, 0, 0] -> majority: 0


## 6. Combined sampling: 50% of the rows and 50% of the columns (2 of 5)

In [8]:
tree, inputs = forest_by_hand(lambda: combined_sampling(df, 0.5, 0.5))
print(export_text(tree, feature_names=list(inputs)))

tree 1: 50 rows (12 repeats), columns ['col2', 'col1'], depth 4, vote 0


tree 2: 50 rows (10 repeats), columns ['col5', 'col4'], depth 5, vote 0
tree 3: 50 rows (14 repeats), columns ['col4', 'col3'], depth 6, vote 1
votes: [0, 0, 1] -> majority: 0
|--- col3 <= 0.33
|   |--- col4 <= 0.85
|   |   |--- col3 <= -0.85
|   |   |   |--- class: 0
|   |   |--- col3 >  -0.85
|   |   |   |--- col4 <= -0.72
|   |   |   |   |--- col4 <= -0.96
|   |   |   |   |   |--- class: 1
|   |   |   |   |--- col4 >  -0.96
|   |   |   |   |   |--- class: 0
|   |   |   |--- col4 >  -0.72
|   |   |   |   |--- col4 <= 0.17
|   |   |   |   |   |--- class: 1
|   |   |   |   |--- col4 >  0.17
|   |   |   |   |   |--- col4 <= 0.34
|   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |--- col4 >  0.34
|   |   |   |   |   |   |--- class: 1
|   |--- col4 >  0.85
|   |   |--- class: 1
|--- col3 >  0.33
|   |--- col4 <= -1.39
|   |   |--- class: 0
|   |--- col4 >  -1.39
|   |   |--- col4 <= -1.00
|   |   |   |--- class: 1
|   |   |--- col4 >  -1.00
|   |   |   |--- class: 0



## 7. The same in one line with scikit-learn
`RandomForestClassifier` does all of this for us, with 100 trees by default.

In [9]:
rf = RandomForestClassifier(random_state=42)
rf.fit(df.iloc[:, :5], df["target"])
print("number of trees:", len(rf.estimators_))
print("prediction for the query:", rf.predict(query.iloc[:, :5]))

number of trees: 100
prediction for the query: [0]


## 8. The query point held out
Observation 7 is one of the 100 training rows, so a tree that drew it has already seen its answer. Here we train on the other 99 rows only and ask again.

In [10]:
train = df.drop(index=7)                      # every row except the query
rng = np.random.default_rng(4)                # same seed and draw order as above
for name, make in [("row sampling", lambda: sample_rows(train, 0.2)),
                   ("column sampling", lambda: sample_features(train, 0.8)),
                   ("combined sampling", lambda: combined_sampling(train, 0.5, 0.5))]:
    votes = []
    for _ in range(3):
        d = make()
        inputs = d.columns[:-1]
        tree = DecisionTreeClassifier(random_state=0).fit(d[inputs], d["target"])
        votes.append(int(tree.predict(query[inputs])[0]))
    print(name, "votes:", votes)
rf_out = RandomForestClassifier(random_state=42).fit(train.iloc[:, :5], train["target"])
tree_votes = [int(t.predict(query.iloc[:, :5].to_numpy())[0]) for t in rf_out.estimators_]
print("100 trees, votes for class 0 and 1:", np.bincount(tree_votes, minlength=2),
      "-> forest predicts", rf_out.predict(query.iloc[:, :5])[0])

row sampling votes: [0, 0, 0]
column sampling votes: [0, 0, 0]
combined sampling votes: [0, 0, 0]


100 trees, votes for class 0 and 1: [89 11] -> forest predicts 0
